In [39]:
import robotic as ry
import numpy as np
import open3d as o3d
     

base_path = "/mnt/c/Users/user/Documents/CS549/grasp_gen/environments/"
base_path_depth = "/mnt/c/Users/user/Documents/CS549/grasp_gen/depth_points/"
base_path_sam = "/mnt/c/Users/user/Documents/CS549/grasp_gen/segmentation_points/"

file_num = 2
env_file_name = f"env_{file_num}.g"
g_file_path = base_path + env_file_name

depth_pts_file_name = f"env_{file_num}_points.npy"
depth_clr_file_name = f"env_{file_num}_colors.npy"
depth_pts_path = base_path_depth + depth_pts_file_name
depth_clr_path = base_path_depth + depth_clr_file_name

sam_pts_file_name = f"env_{file_num}_points.npy"
sam_clr_file_name = f"env_{file_num}_colors.npy"
sam_pts_path = base_path_sam + sam_pts_file_name
sam_clr_path = base_path_sam + sam_clr_file_name
     

C = ry.Config()

C.addFile(g_file_path)

all_frames = C.getFrameNames()
print(all_frames)

#C.view()
     

goal_frames = [f for f in all_frames if "goal" in f]
goal_base_frame = [f for f in goal_frames if "base" in f][0]
goal_prefix = goal_base_frame.replace("_base", "")

goal_cubes = [
    f for f in goal_frames if f.startswith(goal_prefix) and "cube" in f
]
goal_cubes.sort()
print(goal_cubes)
     

goal_pose_frames = [f for f in all_frames if "goal_pose" in f]
goal_pose_base_frame = [f for f in goal_pose_frames if "base" in f][0]
goal_pose_prefix = goal_pose_base_frame.replace("_base", "")

goal_pose_cubes = [
    f for f in goal_pose_frames if f.startswith(goal_pose_prefix) and "cube" in f
]
goal_pose_cubes.sort()

print(goal_pose_cubes)
     

def calculate_centroid(prefix, full_frame_list):
    cubes = [
        f for f in full_frame_list if f.startswith(prefix) and "cube" in f
    ]
    if not cubes:
        return None
    positions = [C.getFrame(f).getPosition() for f in cubes]
    return np.mean(positions, axis=0)
     

goal_centroid = calculate_centroid(goal_prefix, goal_frames)
goal_pose_centroid = calculate_centroid(goal_pose_prefix, goal_pose_frames)

print("Goal Centroid:", goal_centroid)
print("Goal Pose Centroid:", goal_pose_centroid)
     

#pts = np.load(depth_pts_path)
#colors = np.load(depth_clr_path)

pts = np.load(sam_pts_path)
colors = np.load(sam_clr_path)
     

def filter_background_by_coordinates(pts, colors, x_range=(-1.5, 1.5), y_range=(-1.5, 1.5), z_range=(-0.5, 2.0)):
    mask_x = (pts[:, 0] >= x_range[0]) & (pts[:, 0] <= x_range[1])
    mask_y = (pts[:, 1] >= y_range[0]) & (pts[:, 1] <= y_range[1])
    mask_z = (pts[:, 2] >= z_range[0]) & (pts[:, 2] <= z_range[1])
    
    spatial_mask = mask_x & mask_y & mask_z
    
    filtered_pts = pts[spatial_mask]
    filtered_colors = colors[spatial_mask]
    
    print(f"Original points: {pts.shape[0]} | Points kept: {filtered_pts.shape[0]}")
    
    return filtered_pts, filtered_colors
     

x_bounds = (-1.5, 1.5)
y_bounds = (-1.5, 1.5)
z_bounds = (0.65, 2.0)

filtered_pts, filtered_colors = filter_background_by_coordinates(
    pts, colors, x_range=x_bounds, y_range=y_bounds, z_range=z_bounds
)

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(filtered_pts[:, :3])

if filtered_colors.max() > 1.0:
    pcd.colors = o3d.utility.Vector3dVector(filtered_colors[:, :3] / 255.0)
else:
    pcd.colors = o3d.utility.Vector3dVector(filtered_colors[:, :3])

#o3d.visualization.draw_geometries([pcd], window_name="Spatially Cropped Point Cloud")
     

def extract_points_by_centroid(pts, colors, centroid, radius=0.15):
    distances = np.linalg.norm(pts - centroid, axis=1)
    
    mask = distances <= radius
    
    extracted_pts = pts[mask]
    extracted_colors = colors[mask]
    
    print(f"Centroid: {centroid}")
    print(f"Points within {radius}m radius: {extracted_pts.shape[0]}")
    
    return extracted_pts, extracted_colors, distances[mask]
     

radius = 0.15  # Adjust radius as needed

goal_pts, goal_colors, goal_distances = extract_points_by_centroid(
    filtered_pts, filtered_colors, goal_centroid, radius=radius
)

goal_pose_pts, goal_pose_colors, goal_pose_distances = extract_points_by_centroid(
    filtered_pts, filtered_colors, goal_pose_centroid, radius=radius
)
     

goal_pcd = o3d.geometry.PointCloud()
goal_pcd.points = o3d.utility.Vector3dVector(goal_pts[:, :3])
if goal_colors.max() > 1.0:
    goal_pcd.colors = o3d.utility.Vector3dVector(goal_colors[:, :3] / 255.0)
else:
    goal_pcd.colors = o3d.utility.Vector3dVector(goal_colors[:, :3])

goal_pose_pcd = o3d.geometry.PointCloud()
goal_pose_pcd.points = o3d.utility.Vector3dVector(goal_pose_pts[:, :3])
if goal_pose_colors.max() > 1.0:
    goal_pose_pcd.colors = o3d.utility.Vector3dVector(goal_pose_colors[:, :3] / 255.0)
else:
    goal_pose_pcd.colors = o3d.utility.Vector3dVector(goal_pose_colors[:, :3])

# Visualize both point clouds with centroids
geometries = [goal_pcd, goal_pose_pcd]
o3d.visualization.draw_geometries(geometries, window_name="Goal and Goal Pose Point Clouds (Red: Goal, Green: Goal Pose)")          # Comment it out if you don't want to visualize here
     

['world', 'table', 'obj0_base', 'obj0_cube0', 'obj0_joint1', 'obj0_cube1', 'obj0_joint2', 'obj0_cube2', 'obj0_joint3', 'obj0_cube3', 'obj0_joint4', 'obj0_cube4', 'obj0_joint5', 'obj0_cube5', 'obj1_base', 'obj1_cube0', 'obj1_joint1', 'obj1_cube1', 'obj1_joint2', 'obj1_cube2', 'obj1_joint3', 'obj1_cube3', 'obj1_joint4', 'obj1_cube4', 'obj1_joint5', 'obj1_cube5', 'obj2_base', 'obj2_cube0', 'obj2_joint1', 'obj2_cube1', 'obj2_joint2', 'obj2_cube2', 'obj2_joint3', 'obj2_cube3', 'obj2_joint4', 'obj2_cube4', 'obj2_joint5', 'obj2_cube5', 'obj3_base', 'obj3_cube0', 'obj3_joint1', 'obj3_cube1', 'obj3_joint2', 'obj3_cube2', 'obj3_joint3', 'obj3_cube3', 'obj3_joint4', 'obj3_cube4', 'obj3_joint5', 'obj3_cube5', 'obj4_base', 'obj4_cube0', 'obj4_joint1', 'obj4_cube1', 'obj4_joint2', 'obj4_cube2', 'obj4_joint3', 'obj4_cube3', 'obj4_joint4', 'obj4_cube4', 'obj4_joint5', 'obj4_cube5', 'obj5_base', 'obj5_cube0', 'obj5_joint1', 'obj5_cube1', 'obj5_joint2', 'obj5_cube2', 'obj5_joint3', 'obj5_cube3', 'obj5_j

In [40]:
import numpy as np
import open3d as o3d


def align_normals_with_centroid(point_cloud):
    normals = np.asarray(point_cloud.normals)
    pts = np.asarray(point_cloud.points)

    centroid = np.mean(pts, axis=0)
    for i in range(len(normals)):
        to_point = pts[i] - centroid
        if np.dot(normals[i], to_point) < 0:
            normals[i] = -normals[i]

    point_cloud.normals = o3d.utility.Vector3dVector(normals)
    return point_cloud


def reorient_normals_locally(point_cloud):
    normals = np.asarray(point_cloud.normals)
    pts = np.asarray(point_cloud.points)
    kdtree = o3d.geometry.KDTreeFlann(point_cloud)

    for i in range(len(normals)):
        _, idx, _ = kdtree.search_knn_vector_3d(pts[i], 10)
        avg_normal = np.mean(normals[idx], axis=0)
        norm = np.linalg.norm(avg_normal)

        if norm > 1e-12:
            avg_normal = avg_normal / norm
            if np.dot(normals[i], avg_normal) < 0:
                normals[i] = -normals[i]

    point_cloud.normals = o3d.utility.Vector3dVector(normals)
    return point_cloud


# goal_pcd'nin daha önce oluşturulmuş olduğunu varsayıyoruz
# örn: goal_pcd = ...

target_pcd = o3d.geometry.PointCloud(goal_pcd)

points = np.asarray(target_pcd.points)
assert len(points) > 0, "goal_pcd boş geldi"

print("\nLoaded target points shape:", points.shape)
print(points[:5])

colors = None
if target_pcd.has_colors():
    colors = np.asarray(target_pcd.colors)
    print("\nLoaded target colors shape:", colors.shape)
    print(colors[:5])

# raw target object visualization
o3d.visualization.draw_geometries([target_pcd], window_name="Raw Target Object")

# downsample
pcd_tmp = o3d.geometry.PointCloud(target_pcd)
pcd_tmp = pcd_tmp.voxel_down_sample(voxel_size=0.005)

print("\nDownsampled point count:", len(pcd_tmp.points))
if pcd_tmp.has_colors():
    print("Downsampled color count:", len(pcd_tmp.colors))

# estimate normals
pcd_tmp.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.05, max_nn=30)
)

pcd_tmp = align_normals_with_centroid(pcd_tmp)
pcd_tmp = reorient_normals_locally(pcd_tmp)

normals = np.asarray(pcd_tmp.normals)

print("Number of processed points:", len(pcd_tmp.points))
print("Number of normals:", len(normals))

o3d.visualization.draw_geometries(
    [pcd_tmp],
    window_name="Target Object with Normals",
    point_show_normal=True
)


Loaded target points shape: (47316, 3)
[[0.38868743 0.29662988 0.69003296]
 [0.38496793 0.2957     0.69003296]
 [0.3858978  0.2957     0.69003296]
 [0.38775755 0.2957     0.69003296]
 [0.38868743 0.2957     0.69003296]]

Loaded target colors shape: (47316, 3)
[[1. 0. 1.]
 [1. 0. 1.]
 [1. 0. 1.]
 [1. 0. 1.]
 [1. 0. 1.]]

Downsampled point count: 2106
Downsampled color count: 2106
Number of processed points: 2106
Number of normals: 2106


In [18]:
def safe_norm(v, eps=1e-9):
    n = np.linalg.norm(v)
    if n < eps:
        return v
    return v / n


def angle_between(v1, v2):
    v1_u = safe_norm(v1)
    v2_u = safe_norm(v2)
    dot_product = np.dot(v1_u, v2_u)
    angle = np.arccos(np.clip(dot_product, -1.0, 1.0))
    return np.degrees(angle)


def compute_approach_clearance(center, approach_dir, all_points,
                               corridor_length=0.08,
                               corridor_radius=0.015,
                               ignore_radius=0.015):

    approach_dir = safe_norm(approach_dir)

    relative = all_points - center

    # candidate center'ın arkasındaki approach direction boyunca projection
    projection = relative @ approach_dir

    # grasp center'a çok yakın noktaları ignore ediyoruz,
    # çünkü onlar zaten grasp yapılacak objenin kendi noktaları
    valid_depth = (projection > ignore_radius) & (projection < corridor_length)

    perpendicular = relative - np.outer(projection, approach_dir)
    perpendicular_distance = np.linalg.norm(perpendicular, axis=1)

    inside_corridor = valid_depth & (perpendicular_distance < corridor_radius)

    blocking_count = np.sum(inside_corridor)

    # az blocking point iyidir, çok point kötüdür
    clearance_score = np.exp(-0.15 * blocking_count)

    return clearance_score, blocking_count
def compute_pca_axes(points):
    centered = points - np.mean(points, axis=0)

    covariance = np.cov(centered.T)

    eigenvalues, eigenvectors = np.linalg.eigh(covariance)

    # büyükten küçüğe sırala
    order = np.argsort(eigenvalues)[::-1]

    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]

    long_axis = eigenvectors[:, 0]
    mid_axis = eigenvectors[:, 1]
    short_axis = eigenvectors[:, 2]

    return long_axis, mid_axis, short_axis, eigenvalues

def score_candidate(p1, p2, n1, n2, centroid, target_points, obstacle_points, pca_axes,
                    min_distance=0.01,
                    max_distance=0.09,
                    ideal_width=0.05):

    center = (p1 + p2) / 2
    width = np.linalg.norm(p2 - p1)
    grasp_axis = safe_norm(p2 - p1)

    normal_angle = angle_between(n1, n2)
    angle_n1_axis = angle_between(n1, grasp_axis)
    angle_n2_axis = angle_between(n2, grasp_axis)

    reasons = []

    antipodal_score = np.clip((normal_angle - 160) / 20, 0, 1)

    projection_n1 = np.dot(n1, grasp_axis)
    projection_n2 = np.dot(n2, grasp_axis)
    alignment_score = np.clip(0.5 * ((-projection_n1) + projection_n2), 0, 1)

    width_error = abs(width - ideal_width)
    distance_score = np.clip(1 - width_error / (max_distance - min_distance), 0, 1)

    dist_to_centroid = np.linalg.norm(center - centroid)
    centroid_score = np.exp(-20 * dist_to_centroid)
    long_axis, mid_axis, short_axis, eigenvalues = pca_axes

    # grasp axis objenin uzun eksenine değil, daha çok kısa/orta eksenine paralel olursa daha iyi
    align_long = abs(np.dot(grasp_axis, long_axis))
    align_mid = abs(np.dot(grasp_axis, mid_axis))
    align_short = abs(np.dot(grasp_axis, short_axis))

    pca_score = max(align_mid, align_short)
    approach_dir = safe_norm(-(n1 + n2))

    if np.linalg.norm(approach_dir) < 1e-6:
        approach_dir = np.array([0.0, 0.0, 1.0])

    self_clearance_score, self_blocking_count = compute_approach_clearance(
        center,
        approach_dir,
        target_points,
        corridor_length=0.08,
        corridor_radius=0.015,
        ignore_radius=0.015
    )

    scene_clearance_score, scene_blocking_count = compute_approach_clearance(
        center,
        approach_dir,
        obstacle_points,
        corridor_length=0.08,
        corridor_radius=0.015,
        ignore_radius=0.015
    )

    if antipodal_score > 0.8:
        reasons.append("good antipodal normals")
    else:
        reasons.append("weaker antipodal quality")

    if alignment_score > 0.8:
        reasons.append("good normal-axis alignment")
    else:
        reasons.append("weaker normal-axis alignment")

    if distance_score > 0.8:
        reasons.append("good grasp width")
    else:
        reasons.append("less ideal grasp width")

    if dist_to_centroid < 0.03:
        reasons.append("near object centroid")
    else:
        reasons.append("farther from object centroid")
    if pca_score > 0.8:
        reasons.append("good alignment with object minor axis")
    elif align_long > 0.8:
        reasons.append("grasp axis aligned with long object axis")
    else:
        reasons.append("moderate PCA alignment")
    if self_blocking_count == 0:
        reasons.append("clear self approach")
    elif self_blocking_count < 10:
        reasons.append("partially blocked by target geometry")
    else:
        reasons.append("blocked by target geometry")

    if scene_blocking_count == 0:
        reasons.append("clear scene approach")
    elif scene_blocking_count < 10:
        reasons.append("partially blocked by scene clutter")
    else:
        reasons.append("blocked by scene clutter")

    score = (
        0.22 * antipodal_score +
        0.18 * alignment_score +
        0.13 * distance_score +
        0.08 * centroid_score +
        0.12 * pca_score +
        0.10 * self_clearance_score +
        0.17 * scene_clearance_score
    )

    return {
        "score": score,
        "center": center,
        "width": width,
        "normal_angle": normal_angle,
        "angle_n1_axis": angle_n1_axis,
        "angle_n2_axis": angle_n2_axis,
        "dist_to_centroid": dist_to_centroid,
        "antipodal_score": antipodal_score,
        "alignment_score": alignment_score,
        "distance_score": distance_score,
        "centroid_score": centroid_score,
        "pca_score": pca_score,
        "align_long": align_long,
        "align_mid": align_mid,
        "align_short": align_short,
        "self_clearance_score": self_clearance_score,
        "self_blocking_count": self_blocking_count,
        "scene_clearance_score": scene_clearance_score,
        "scene_blocking_count": scene_blocking_count,
        "approach_dir": approach_dir,
        "reasons": reasons
    }

def create_grasp_visualization(candidate, color=[0, 1, 0], radius=0.002):
    p1 = candidate[0]
    p2 = candidate[1]
    n1 = candidate[2]
    n2 = candidate[3]

    geometries = []

    sphere1 = o3d.geometry.TriangleMesh.create_sphere(radius=radius)
    sphere1.translate(p1)
    sphere1.paint_uniform_color(color)

    sphere2 = o3d.geometry.TriangleMesh.create_sphere(radius=radius)
    sphere2.translate(p2)
    sphere2.paint_uniform_color(color)

    line = o3d.geometry.LineSet()
    line.points = o3d.utility.Vector3dVector([p1, p2])
    line.lines = o3d.utility.Vector2iVector([[0, 1]])
    line.colors = o3d.utility.Vector3dVector([color])

    normal_line1 = o3d.geometry.LineSet()
    normal_line1.points = o3d.utility.Vector3dVector([p1, p1 + 0.03 * n1])
    normal_line1.lines = o3d.utility.Vector2iVector([[0, 1]])
    normal_line1.colors = o3d.utility.Vector3dVector([color])

    normal_line2 = o3d.geometry.LineSet()
    normal_line2.points = o3d.utility.Vector3dVector([p2, p2 + 0.03 * n2])
    normal_line2.lines = o3d.utility.Vector2iVector([[0, 1]])
    normal_line2.colors = o3d.utility.Vector3dVector([color])

    geometries.extend([sphere1, sphere2, line, normal_line1, normal_line2])
    return geometries


min_distance = 0.01
max_distance = 0.09
small_angle_threshold = 10
large_angle_threshold = 160

top_k = 10

best_grasps = []
compute_grasp_points = True
cropped_pcl = np.asarray(pcd_tmp.points)

if compute_grasp_points:
    points = cropped_pcl
    grasp_candidates = []
    scored_candidates = []

    centroid = np.mean(points, axis=0)
    scene_points = filtered_pts[:, :3]

    dist_to_target_centroid = np.linalg.norm(scene_points - centroid, axis=1)

    obstacle_points = scene_points[dist_to_target_centroid > radius]

    obstacle_pcd = o3d.geometry.PointCloud()
    obstacle_pcd.points = o3d.utility.Vector3dVector(obstacle_points)

    obstacle_pcd = obstacle_pcd.voxel_down_sample(voxel_size=0.02)

    obstacle_points = np.asarray(obstacle_pcd.points)

    target_points = points
    pca_axes = compute_pca_axes(target_points)

    long_axis, mid_axis, short_axis, eigenvalues = pca_axes

    print("PCA eigenvalues:", eigenvalues)
    print("Long axis:", long_axis)
    print("Mid axis:", mid_axis)
    print("Short axis:", short_axis)

    print("Scene points:", len(scene_points))
    print("Target points:", len(target_points))
    print("Obstacle points used for scene clearance:", len(obstacle_points))


    for i in range(len(points)):
        for j in range(i + 1, len(points)):
            p1, p2 = points[i], points[j]
            n1, n2 = normals[i], normals[j]

            distance = np.linalg.norm(p1 - p2)

            if min_distance <= distance <= max_distance:
                angle = angle_between(n1, n2)

                if large_angle_threshold <= angle <= 180:
                    epiline = safe_norm(p2 - p1)

                    projection_n1 = np.dot(n1, epiline)
                    projection_n2 = np.dot(n2, epiline)

                    if projection_n1 < 0 and projection_n2 > 0:
                        angle_n1_epiline = angle_between(n1, epiline)
                        angle_n2_epiline = angle_between(n2, epiline)

                        if (
                            min(abs(angle_n1_epiline), abs(angle_n2_epiline)) <= small_angle_threshold
                            and max(abs(angle_n1_epiline), abs(angle_n2_epiline)) >= large_angle_threshold
                        ):
                            info = score_candidate(
                                p1, p2, n1, n2, centroid, target_points, obstacle_points, pca_axes,
                                min_distance=min_distance,
                                max_distance=max_distance,
                                ideal_width=0.05
                            )

                            candidate = (
                                p1,
                                p2,
                                n1,
                                n2,
                                info["normal_angle"],
                                info["width"],
                                info["angle_n1_axis"],
                                info["angle_n2_axis"],
                                info["dist_to_centroid"],
                                info["score"],
                                info["reasons"],
                                info["self_clearance_score"],
                                info["self_blocking_count"],
                                info["scene_clearance_score"],
                                info["scene_blocking_count"],
                                info["pca_score"],
                                info["align_long"],
                                info["align_mid"],
                                info["align_short"]
                            )

                            grasp_candidates.append(candidate)
                            scored_candidates.append(candidate)

    if scored_candidates:
        scored_candidates = sorted(
            scored_candidates,
            key=lambda x: x[9],
            reverse=True
        )

        print("Number of valid grasp candidates:", len(scored_candidates))
        print("\nTop candidates:")

        for rank, cand in enumerate(scored_candidates[:top_k]):
            print("\nCandidate", rank)
            print("score:", round(cand[9], 4))
            print("width:", round(cand[5], 4))
            print("normal_angle:", round(cand[4], 2))
            print("dist_to_centroid:", round(cand[8], 4))
            print("self_clearance_score:", round(cand[11], 4))
            print("self_blocking_count:", cand[12])
            print("scene_clearance_score:", round(cand[13], 4))
            print("scene_blocking_count:", cand[14])
            print("pca_score:", round(cand[15], 4))
            print("align_long:", round(cand[16], 4))
            print("align_mid:", round(cand[17], 4))
            print("align_short:", round(cand[18], 4))
            print("reasons:", cand[10])


        blocked_candidates = [c for c in scored_candidates if c[12] > 0 or c[14] > 0]

        print("\nNumber of blocked candidates:", len(blocked_candidates))

        if blocked_candidates:
            print("\nExample blocked candidates:")

            for rank, cand in enumerate(blocked_candidates[:5]):
                print("\nBlocked Candidate", rank)
                print("score:", round(cand[9], 4))
                print("width:", round(cand[5], 4))
                print("normal_angle:", round(cand[4], 2))
                print("dist_to_centroid:", round(cand[8], 4))
                print("self_clearance_score:", round(cand[11], 4))
                print("self_blocking_count:", cand[12])
                print("scene_clearance_score:", round(cand[13], 4))
                print("scene_blocking_count:", cand[14])
                print("pca_score:", round(cand[15], 4))
                print("align_long:", round(cand[16], 4))
                print("align_mid:", round(cand[17], 4))
                print("align_short:", round(cand[18], 4))
                print("reasons:", cand[10])

        best_grasp = scored_candidates[0]
        best_grasps.append(best_grasp)
        # Visualize top-k grasp candidates
        top_k_vis = 5

        object_cloud = o3d.geometry.PointCloud()
        object_cloud.points = o3d.utility.Vector3dVector(cropped_pcl)
        object_cloud.paint_uniform_color([0.6, 0.6, 0.6])

        visual_geometries = [object_cloud]

        colors = [
            [0, 1, 0],      # best: green
            [0, 0, 1],      # blue
            [1, 0.7, 0],    # orange
            [1, 0, 1],      # purple
            [0, 1, 1]       # cyan
        ]

        for rank, cand in enumerate(scored_candidates[:top_k_vis]):
            visual_geometries.extend(
                create_grasp_visualization(
                    cand,
                    color=colors[rank % len(colors)],
                    radius=0.0025 if rank == 0 else 0.0018
                )
            )

        o3d.visualization.draw_geometries(
            visual_geometries,
            window_name="Top-k Grasp Candidates"
        )

        print("\nSelected best grasp:")
        print(best_grasp)

    else:
        print("No suitable grasp candidates found")
        grasp_candidates.append(None)

PCA eigenvalues: [0.0031465  0.00127535 0.00041615]
Long axis: [-0.98759341 -0.11180122 -0.11027122]
Mid axis: [ 0.0548441  -0.90356832  0.42491932]
Short axis: [-0.14714408  0.41359979  0.89848975]
Scene points: 531628
Target points: 2106
Obstacle points used for scene clearance: 1360
Number of valid grasp candidates: 3191

Top candidates:

Candidate 0
score: 0.9521
width: 0.0499
normal_angle: 179.81
dist_to_centroid: 0.0303
self_clearance_score: 1.0
self_blocking_count: 0
scene_clearance_score: 1.0
scene_blocking_count: 0
pca_score: 0.9326
align_long: 0.1042
align_mid: 0.9326
align_short: 0.3455
reasons: ['good antipodal normals', 'good normal-axis alignment', 'good grasp width', 'farther from object centroid', 'good alignment with object minor axis', 'clear self approach', 'clear scene approach']

Candidate 1
score: 0.951
width: 0.0502
normal_angle: 179.91
dist_to_centroid: 0.0276
self_clearance_score: 1.0
self_blocking_count: 0
scene_clearance_score: 1.0
scene_blocking_count: 0
pca

In [36]:
if compute_grasp_points:
    best_grasps_np = np.array(best_grasps, dtype=object)
    # grasp_candidates_np = np.array(grasp_candidates, dtype=object)

    np.savez("best_grasps.npz", best_grasps=best_grasps_np)
    # np.savez("grasp_candidates.npz", grasp_candidates=grasp_candidates_np)
else:
    loaded_data = np.load("best_grasps.npz", allow_pickle=True)
    best_grasps = loaded_data['best_grasps']
    # grasp_candidates = loaded_data['grasp_candidates']

In [37]:
for i, best_grasp in enumerate(best_grasps):


    p1 = best_grasp[0]
    p2 = best_grasp[1]
    n1 = best_grasp[2]
    n2 = best_grasp[3]

    # diplay selected points and their normals in open3d
    point_cloud = o3d.geometry.PointCloud()

    point_cloud.points = o3d.utility.Vector3dVector(cropped_pcl)
    point_cloud.colors = o3d.utility.Vector3dVector(
        np.tile([0.5, 0.5, 0.5], (len(cropped_pcl), 1))
    )

    normals = np.array([n1, n2])
    point_cloud.normals = o3d.utility.Vector3dVector(normals)

    grasp_visualizations = []

    sphere1 = o3d.geometry.TriangleMesh.create_sphere(radius=0.001)
    sphere1.translate(p1)
    sphere1.paint_uniform_color([1, 0, 0])  

    sphere2 = o3d.geometry.TriangleMesh.create_sphere(radius=0.001)
    sphere2.translate(p2)
    sphere2.paint_uniform_color([0, 1, 0]) 

    # Add spheres to the visualization
    grasp_visualizations.append(sphere1)
    grasp_visualizations.append(sphere2)

    # Create lines to represent normals at each grasp point
    normal_line1 = o3d.geometry.LineSet()
    normal_line2 = o3d.geometry.LineSet()

    # Normal vectors n1 and n2 at points p1 and p2 respectively

    line_points1 = [p1, p1 + 0.05 * n1]  # Line representing normal direction for p1
    line_points2 = [p2, p2 + 0.05 * n2]  # Line representing normal direction for p2
   
    normal_line1.points = o3d.utility.Vector3dVector(line_points1)
    normal_line1.lines = o3d.utility.Vector2iVector([[0, 1]])  # Line from p1 to p1 + normal
    normal_line1.colors = o3d.utility.Vector3dVector([[1, 0, 0]])  # Color red

    normal_line2.points = o3d.utility.Vector3dVector(line_points2)
    normal_line2.lines = o3d.utility.Vector2iVector([[0, 1]])  # Line from p2 to p2 + normal
    normal_line2.colors = o3d.utility.Vector3dVector([[0, 1, 0]])  # Color green

    # Add normal lines to the visualization
    grasp_visualizations.append(normal_line1)
    grasp_visualizations.append(normal_line2)

    o3d.visualization.draw_geometries([point_cloud] + grasp_visualizations)

